# hihAigua: kaynak ve kalite incelemesi

Kaynak birimi doğrulanmadan litre dönüşümü ve kanonik yayın kapalıdır. Bu notebook model eğitmez; soğuk/sıcak suyu ayrı gösterir.

In [ ]:
from pathlib import Path
import json
import pyarrow.parquet as pq
from urban_water.sources import hihaigua_reference as reference

root = reference.PROJECT_ROOT
pointer = json.loads((root / 'var/research/hihaigua/v2/latest.json').read_text(encoding='utf-8'))
directory = Path(pointer['directory'])
assert reference.sha256(directory / 'manifest.json') == pointer['manifest_sha256']
manifest = reference.verify_prepared(directory, reference.sha256(reference.POLICY))
assert manifest is not None
{'directory': str(directory), 'field_households': manifest['field_households'], 'model_ready': manifest['model_ready']}


In [ ]:
summary = []
for component, data in manifest['components'].items():
    path = directory / component / 'counters.parquet'
    assert pq.read_metadata(path).num_rows == data['rows']
    summary.append({'component': component, 'rows': data['rows'], 'streams': len(data['streams']),
                    'canonical_ready': data['canonical_ready'], 'quality_counts': data['quality_counts']})
summary


In [ ]:
columns = ['household_code', 'occurred_at', 'received_at', 'value_source',
           'counter_value', 'contract_value', 'unit', 'delta_status', 'quality_flags']
first_batch = next(pq.ParquetFile(directory / 'hotWater/counters.parquet').iter_batches(batch_size=5, columns=columns))
first_batch.to_pylist()


`value_source` özgün metindir. `counter_value` bu değerin hassas ondalık karşılığıdır. `contract_value` dokuz ondalık basamağa göre ayrı hazırlanır; farkı kaynak tablosunda saklanır.

`received_at` gerçek arşiv indirme zamanıdır. Tarihsel cihaz erişim zamanı değildir. Endeks düşüşü ve gözlem boşluğu kaçak etiketi sayılmaz.

Yedi dakikalık örneklerin saat/gün sınırlarını kesmesi nedeniyle kanonik özetler kısmi olabilir. Tam günlük hedef politikası ayrıca belirlenmelidir. Kaynak birimi `research/reference_data/hihaigua-v2.json` içinde kanıtla doğrulanmadan `sources.toml` üretilmez.